# Week 16 Optional: AgentCore Runtime Deployment

## Overview

This optional notebook covers advanced AgentCore topics for students who want
to go deeper into production agent deployment. It is **NOT required** for the course.

## What You'll Learn

1. **AgentCore Runtime** — Package and deploy a Strands agent to managed containers
2. **AgentCore Gateway** — Convert existing APIs into MCP-compatible tools
3. **Human-in-the-Loop** — Add approval gates for high-risk decisions
4. **Advanced Multi-Agent Patterns** — Supervisor vs Arbiter, parallel execution

## Prerequisites

- Completed Week 16 main notebook (Strands agents, multi-agent, AgentCore Memory)
- AWS credentials with AgentCore permissions

In [ ]:
# =============================================================================
# SETUP
# =============================================================================

!pip install -q strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit

import os
import json
import boto3
from getpass import getpass
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore import BedrockAgentCoreApp

# AWS credentials (same as main notebook)
if not os.environ.get('AWS_ACCESS_KEY_ID'):
    os.environ['AWS_ACCESS_KEY_ID'] = getpass("AWS Access Key ID: ")
    os.environ['AWS_SECRET_ACCESS_KEY'] = getpass("AWS Secret Access Key: ")
    os.environ['AWS_DEFAULT_REGION'] = input("AWS Region [us-east-1]: ") or 'us-east-1'

sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"✅ Authenticated as: {identity['Arn']}")

# Section 1: AgentCore Runtime Deployment

## What is AgentCore Runtime?

AgentCore Runtime hosts your agent code in **managed containers**. You package
your Strands agent as a Python application, deploy it, and AWS handles:
- Container orchestration and scaling
- Load balancing across requests
- Health checks and auto-recovery
- Secure networking and IAM integration

## The Deployment Pattern

```python
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# 1. Create your app
app = BedrockAgentCoreApp()

# 2. Define your agent as the entrypoint
@app.entrypoint
def handle_request(user_input: str) -> str:
    agent = Agent(
        model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0"),
        tools=[my_tool_1, my_tool_2],
        system_prompt="You are a fraud investigator."
    )
    result = agent(user_input)
    return str(result)
```

Then deploy with the CLI:
```bash
agentcore configure -e my_agent.py
agentcore launch
```

After deployment, invoke via boto3:
```python
client = boto3.client('bedrock-agentcore')
response = client.invoke_agent_runtime(
    agentRuntimeArn="arn:aws:bedrock-agentcore:us-east-1:123456:runtime/my-agent",
    input={"text": "Investigate TXN-001"}
)
```

In [ ]:
# =============================================================================
# DEMO: Package a Fraud Agent for AgentCore Runtime
# =============================================================================
# We'll create the agent entrypoint file that AgentCore Runtime will execute.
# Note: actual deployment requires CLI commands — we'll write the file here
# and show the deploy commands.

# Re-create the fraud tools (same as main notebook)
@tool
def rt_lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.

    Args:
        transaction_id: The transaction ID to look up
    """
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas",
                     "time": "3:47 AM", "location": "Lagos, Nigeria"},
        "TXN-002": {"amount": 89.99, "type": "subscription", "merchant": "Netflix",
                     "time": "6:00 PM", "location": "Chicago, IL"},
    }
    txn = transactions.get(transaction_id)
    return json.dumps(txn, indent=2) if txn else f"{transaction_id} not found."

@tool
def rt_calculate_risk(amount: float, is_international: bool) -> str:
    """Calculate a basic risk score.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction crosses borders
    """
    score = 0
    if amount > 1000: score += 40
    if is_international: score += 40
    level = "LOW" if score < 30 else "MEDIUM" if score < 60 else "HIGH"
    return json.dumps({"risk_score": score, "risk_level": level})

# Write the entrypoint file
entrypoint_code = '''
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
import json

app = BedrockAgentCoreApp()

@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.
    Args:
        transaction_id: The transaction ID to look up
    """
    # In production, this would query a real database
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas"},
    }
    txn = transactions.get(transaction_id)
    return json.dumps(txn) if txn else f"{transaction_id} not found."

@app.entrypoint
def handle_request(user_input: str) -> str:
    agent = Agent(
        model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0"),
        tools=[lookup_transaction],
        system_prompt="You are a fraud investigator. Investigate transactions and provide verdicts."
    )
    result = agent(user_input)
    return str(result)
'''

with open('fraud_agent_runtime.py', 'w') as f:
    f.write(entrypoint_code)

print("✅ Created fraud_agent_runtime.py")
print()
print("To deploy to AgentCore Runtime, run in terminal:")
print("  agentcore configure -e fraud_agent_runtime.py")
print("  agentcore launch")
print()
print("After deployment, invoke via boto3:")
print("  client = boto3.client('bedrock-agentcore')")
print("  response = client.invoke_agent_runtime(")
print("      agentRuntimeArn='<your-arn>',")
print("      input={'text': 'Investigate TXN-001'})")

## Lab 1: Write Your Own AgentCore Entrypoint (20 minutes)

### Your Task

Create a production-ready agent entrypoint that handles the full multi-agent
fraud pipeline from the main notebook (Triage + Investigation + Decision).

### Steps

1. **Write a `multi_agent_runtime.py`** file that:
   - Defines all 4 fraud tools (`lookup_transaction`, `check_customer_history`,
     `calculate_risk_score`, `check_fraud_policy`)
   - Creates 3 specialist agents (triage, investigation, decision)
   - Wraps each as `@tool` for the supervisor
   - Uses `@app.entrypoint` to expose the supervisor agent
2. **Add error handling** — if any sub-agent fails, catch the exception and
   return a structured error response instead of crashing
3. **Test locally** by calling `handle_request()` directly before deploying

### Hints

- Use `try/except` inside each `run_*` wrapper tool
- Return `json.dumps({"error": str(e)})` on failure
- The `@app.entrypoint` function should accept a string and return a string

In [ ]:
# =============================================================================
# LAB 1: MULTI-AGENT AGENTCORE ENTRYPOINT
# =============================================================================

# Write your multi-agent entrypoint code here
multi_agent_code = None  # YOUR CODE — build the full entrypoint as a string

# Write to file
# with open('multi_agent_runtime.py', 'w') as f:
#     f.write(multi_agent_code)

# Test locally (call the function directly)
# from multi_agent_runtime import handle_request
# result = handle_request("Investigate TXN-001 for CUST-001")
# print(result)

# Section 2: AgentCore Gateway — APIs as MCP Tools

## What is AgentCore Gateway?

AgentCore Gateway converts your existing **APIs and Lambda functions** into
**MCP-compatible tools** that any agent can discover and use — no code changes
to the original API.

```
Your existing API          AgentCore Gateway         Your Strands Agent
+------------------+      +------------------+      +------------------+
| REST API         | ---> | Auto-generates   | ---> | Discovers tools  |
| Lambda function  |      | MCP tool specs   |      | via MCP protocol |
| OpenAPI spec     |      | Handles auth     |      | Calls them       |
+------------------+      +------------------+      +------------------+
```

**Why this matters**: In production, your fraud investigation tools won't be
simulated Python functions — they'll be real REST APIs backed by databases.
Gateway lets your agents use those APIs without writing custom tool wrappers.

## Setting Up a Gateway (Console Walkthrough)

1. Open **Amazon Bedrock AgentCore** console
2. Go to **Gateways** > **Create gateway**
3. Add a **target** (Lambda function or API Gateway REST API)
4. Gateway auto-generates MCP tool definitions from your API spec
5. Connect your agent to the Gateway endpoint

## Connecting an Agent to Gateway Tools

```python
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Connect to your Gateway endpoint
async with streamablehttp_client(
    url="https://your-gateway-url.execute-api.us-east-1.amazonaws.com",
    headers={"Authorization": f"Bearer {token}"}
) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # Discover available tools
        tools = await session.list_tools()
        print(f"Available tools: {[t.name for t in tools.tools]}")
```

In [ ]:
# =============================================================================
# DEMO: Simulating Gateway Tool Discovery
# =============================================================================
# Since setting up a real Gateway requires console access, we'll simulate
# how an agent would discover and use tools from a Gateway endpoint.

# Simulate what Gateway returns as MCP tool definitions
gateway_tools = [
    {
        "name": "query_transaction_db",
        "description": "Query the fraud transaction database via REST API",
        "inputSchema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string", "description": "Transaction ID"},
                "include_history": {"type": "boolean", "description": "Include full history"},
            },
            "required": ["transaction_id"]
        }
    },
    {
        "name": "run_ofac_screening",
        "description": "Run OFAC sanctions screening via compliance API",
        "inputSchema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Name to screen"},
                "country": {"type": "string", "description": "Country of origin"},
            },
            "required": ["name", "country"]
        }
    },
]

print("Simulated Gateway Tool Discovery:")
print(f"  Gateway URL: https://abc123.execute-api.us-east-1.amazonaws.com/mcp")
print(f"  Tools discovered: {len(gateway_tools)}")
for t in gateway_tools:
    print(f"    - {t['name']}: {t['description']}")
print()
print("In production, these tools are auto-generated from your REST APIs.")
print("Your agent calls them via MCP protocol — no custom wrapper code needed.")

## Lab 2: Build a Simulated Gateway Integration (15 minutes)

### Your Task

Create a "gateway simulator" — a function that mimics how AgentCore Gateway
converts a REST API response into a tool result that an agent can consume.

### Steps

1. **Create a `simulate_gateway_call` function** that:
   - Takes a tool name and parameters
   - Simulates calling a REST API (use a dict of mock API responses)
   - Returns the response in the MCP tool result format
2. **Wrap it as a Strands `@tool`** so your agent can call it
3. **Create an agent** that uses your gateway-simulated tool alongside the
   regular fraud tools. Test it on TXN-001.

### Expected Output

- A working gateway simulator that handles `query_transaction_db` and `run_ofac_screening`
- An agent that seamlessly mixes local tools and "gateway" tools

In [ ]:
# =============================================================================
# LAB 2: SIMULATED GATEWAY INTEGRATION
# =============================================================================

# Step 1: Create the gateway simulator
@tool
def gateway_query_transaction_db(transaction_id: str, include_history: bool = False) -> str:
    """Query the fraud transaction database via simulated Gateway REST API.

    Args:
        transaction_id: Transaction ID to query
        include_history: Whether to include full transaction history
    """
    result = None  # YOUR CODE

    return json.dumps(result, indent=2)

@tool
def gateway_run_ofac_screening(name: str, country: str) -> str:
    """Run OFAC sanctions screening via simulated Gateway compliance API.

    Args:
        name: Name to screen against OFAC list
        country: Country of origin
    """
    result = None  # YOUR CODE

    return json.dumps(result, indent=2)


# Step 2: Create agent with mixed local + gateway tools
gateway_agent = None  # YOUR CODE


# Step 3: Test
gateway_result = None  # YOUR CODE

print(gateway_result)

# Section 3: Human-in-the-Loop Patterns

## When Agents Need Human Approval

Not every decision should be automated. For HIGH risk fraud cases, you may want
a human to review and approve before the agent takes action (e.g., blocking an
account or filing a regulatory report).

**Pattern**: Add an "approval gate" tool that pauses execution and waits for
human input before continuing.

```python
@tool
def request_human_approval(case_summary: str, proposed_action: str) -> str:
    """Request human approval for a high-risk fraud decision.

    Args:
        case_summary: Summary of the investigation findings
        proposed_action: The action the agent wants to take
    """
    print(f"\n{'='*60}")
    print(f"HUMAN APPROVAL REQUIRED")
    print(f"{'='*60}")
    print(f"Case: {case_summary}")
    print(f"Proposed action: {proposed_action}")
    print(f"{'='*60}")
    decision = input("Approve? (yes/no/modify): ").strip().lower()
    if decision == "yes":
        return json.dumps({"approved": True, "action": proposed_action})
    elif decision == "modify":
        new_action = input("Enter modified action: ")
        return json.dumps({"approved": True, "action": new_action})
    else:
        return json.dumps({"approved": False, "reason": "Human reviewer rejected"})
```

This is a simple synchronous pattern. In production with AgentCore Runtime, you
would use async webhooks or SNS notifications instead of `input()`.

## Lab 3: Add Human-in-the-Loop to Your Pipeline (15 minutes)

### Your Task

Add a human approval gate to the multi-agent fraud pipeline so that HIGH-risk
decisions require human sign-off before execution.

### Steps

1. **Create a `request_human_approval` tool** using the pattern shown above
2. **Update the supervisor's system prompt** to include:
   "If the triage result is HIGH risk, call `request_human_approval` before
   calling `run_decision`. Include the investigation findings in the case summary."
3. **Test with TXN-001** (HIGH risk) — the agent should pause for your approval
4. **Test with TXN-002** (LOW risk) — the agent should skip approval entirely

### Expected Output

- HIGH risk transactions pause for human approval
- LOW risk transactions flow through without interruption
- The final verdict reflects whether human approval was granted or denied

In [ ]:
# =============================================================================
# LAB 3: HUMAN-IN-THE-LOOP PIPELINE
# =============================================================================

# Step 1: Create the approval gate tool
@tool
def request_human_approval(case_summary: str, proposed_action: str) -> str:
    """Request human approval for a high-risk fraud decision.

    Args:
        case_summary: Summary of the investigation findings
        proposed_action: The action the agent wants to take
    """
    result = None  # YOUR CODE

    return json.dumps(result, indent=2)


# Step 2: Create supervisor with approval gate
hitl_supervisor = None  # YOUR CODE


# Step 3: Test with HIGH risk (should ask for approval)
print("Testing HIGH risk transaction (should pause for approval)...")
high_risk_result = None  # YOUR CODE

print(high_risk_result)

# Summary

## What You Explored

1. **AgentCore Runtime** — Package agents as Python apps, deploy to managed containers
2. **AgentCore Gateway** — Convert REST APIs into MCP tools automatically
3. **Human-in-the-Loop** — Add approval gates for high-risk decisions

## Key Takeaways

- AgentCore Runtime uses `@app.entrypoint` to expose your agent as a service
- Gateway eliminates custom tool wrappers — your existing APIs become agent tools
- HITL patterns are essential for regulated industries like financial services
- In production, use async notifications (SNS/EventBridge) instead of `input()`

## Further Reading

- [AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/)
- [Strands Agents SDK](https://strandsagents.com/)
- [AgentCore Samples on GitHub](https://github.com/awslabs/amazon-bedrock-agentcore-samples)